Import Statements

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import gdown
from sklearn.preprocessing import MinMaxScaler

Input data and LSTM parameters/hyperparameters

In [17]:
#Get Dataset

file_id = '1tNHOqvjUowTLfgQE2qiNNGiWCSYt8su_'
url = f'https://google.com/uc?id={file_id}'
output = 'nasdaq_data.csv'
gdown.download(url, output, quiet=False)
dataset = pd.read_csv(output, nrows=5)
print("Actual Columns found in CSV:", dataset.columns.tolist())
#Data preprocessing
dataset['Date'] = pd.to_datetime(dataset['Date'])
dataset = dataset.sort_values(['ticker', 'Date']).reset_index(drop=True)
#Data feature headers
features = [
    'Target_Log_Return', 'day_of_week_sin', 'day_of_week_cos', 
    'close_std', 'volume_std', 'price_lag_1d_std', 'price_lag_3d_std', 
    'rolling_volatility_7d_std', 'close_robust', 'volume_robust', 
    'price_lag_1d_robust', 'price_lag_3d_robust', 'rolling_volatility_7d_robust'
]
data = dataset[features].values

#Split data into training and testing
train_size = int(len(dataset) * 0.8)
train_data = dataset.iloc[:train_size]
test_data = dataset.iloc[train_size:]

#Normalize data
normalize = MinMaxScaler()
normalize.fit(train_data[features])

def scale_data(data, scaler, features):
    scaled_values = scaler.transform(data[features])
    new_data = data.copy()
    new_data[features] = scaled_values
    return new_data

train_scaled_data = scale_data(train_data, normalize, features)
test_scaled_data = scale_data(test_data, normalize, features)

#LSTM model hyperparameters
input_size = len(features) #Number of used input features
hidden_size = 256
num_layers = 3
sequence_length = 5 #Shortest sequence length for polymarket data is 6 days, longest is 26 days
learning_rate = 0.0001
epochs = 200
output_size = len(features) #Number of output features
num_forecasted_days = 7 #Number of days to forecast prices

Create LSTM model

In [ ]:
#Create the model
class LSTM(nn.Module):
    #Initialize the LSTM model
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, output_size=1):
        super(LSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.hidden_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, states = self.lstm(x)
        return self.hidden_layer(output[:, -1, :])

Train LSTM model on data

In [ ]:
lstm = LSTM(input_size, hidden_size, num_layers, output_size=output_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm.parameters(), lr=learning_rate)

def create_grouped_sequences(data, feature_cols, sequence_length):
    xs = []
    ys = []
    data = data.sort_values(['ticker', 'Date'])
    for ticker, group in data.groupby('ticker'):
        group_data = group[feature_cols].values
        if len(group_data) > sequence_length:
            for i in range(len(group_data) - sequence_length):
                xs.append(group_data[i : i + sequence_length])
                ys.append(group_data[i + sequence_length])
    if len(xs) == 0:
        raise ValueError(f"No sequences were created! Check if your markets have at least {sequence_length + 1} rows.")
    xs_np = np.array(xs)
    ys_np = np.array(ys)
    return torch.tensor(xs_np, dtype=torch.float32), torch.tensor(ys_np, dtype=torch.float32)

X_train, y_train = create_grouped_sequences(train_scaled_data, features, sequence_length)
X_test, y_test = create_grouped_sequences(test_scaled_data, features, sequence_length)

train_dataset = TensorDataset(X_train, y_train)
loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

lstm.train()
for epoch in range(epochs):
    lstm.train()
    train_loss = 0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        outputs = lstm(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    lstm.eval() 
    with torch.no_grad():
        test_outputs = lstm(X_test)
        test_loss = criterion(test_outputs, y_test)
    if (epoch + 1) % 10 == 0:
        avg_train_loss = train_loss / len(loader)
        print(f'Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.5f}, Test Loss: {test_loss.item():.5f}')

Epoch [10/50] - Train Loss: 0.00702, Test Loss: 0.00206
Epoch [20/50] - Train Loss: 0.00504, Test Loss: 0.00166
Epoch [30/50] - Train Loss: 0.00460, Test Loss: 0.00183
Epoch [40/50] - Train Loss: 0.00373, Test Loss: 0.00159
Epoch [50/50] - Train Loss: 0.00382, Test Loss: 0.00151


Output data/results/conclusions

In [ ]:
def prediction(model, input_sequence, normalize):
    model.eval()
    with torch.no_grad():
        scaled_prediction = model(input_sequence).cpu().numpy() 
        inverse_array = normalize.inverse_transform(scaled_prediction)
        return inverse_array[:, 0]

def forecast_future(model, last_sequence, scaler, days_to_predict=7):
    model.eval()
    current_sequence = last_sequence.clone()
    predictions_scaled = []
    with torch.no_grad():
        for _ in range(days_to_predict):
            prediction = model(current_sequence)
            predictions_scaled.append(prediction.squeeze().cpu().numpy())
            current_sequence = torch.cat((current_sequence[:, 1:, :], prediction.unsqueeze(1)), dim=1)
    
    full_prediction_array = np.array(predictions_scaled)
    inverse_array = scaler.inverse_transform(full_prediction_array)
    return inverse_array[:, 0]

full_scaled_data = scale_data(dataset, normalize, features)
unique_tickers = dataset['ticker'].unique()
all_forecasts = []

print(f"Generating forecasts for {len(unique_tickers)} tickers...")

for target_id in unique_tickers:
    market_group = full_scaled_data[full_scaled_data['ticker'] == target_id]
    if len(market_group) >= sequence_length:
        recent_data_values = market_group[features].tail(sequence_length).values
        latest_data_sequence = torch.tensor(np.array(recent_data_values), dtype=torch.float32).unsqueeze(0)
        future_percentages = forecast_future(lstm, latest_data_sequence, normalize, num_forecasted_days)
        all_forecasts.append([target_id] + list(future_percentages))

forecast_cols = ['ticker'] + [f'Day_{i+1}' for i in range(num_forecasted_days)]
forecast_df = pd.DataFrame(all_forecasts, columns=forecast_cols)
print("\n--- Final Forecast Results ---")
print(forecast_df.head(10))

forecast_df['Total_Predicted_Return'] = forecast_df['Day_7'] - forecast_df['Day_1']

top_movers = forecast_df.sort_values(by='Total_Predicted_Return', ascending=False)

print("\n--- Top 5 Predicted Price Increases ---")
print(top_movers[['market_id', 'Day_1', 'Day_7', 'Total_Predicted_Return']].head(5))


def forecast_external_market(model, new_market_data, scaler, features):
    """
    new_market_data: A DataFrame with exactly 5 rows and the 19 feature columns
    """
    model.eval()
    scaled_values = scaler.transform(new_market_data[features])
    input_tensor = torch.tensor(np.array(scaled_values), dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        forecast = forecast_future(model, input_tensor, scaler, days_to_predict=7)
    return forecast

# EXAMPLE USAGE:
# new_data = pd.read_csv('new_market_last_5_days.csv')
# prediction = forecast_external_market(lstm, new_data, normalize, features)
# print(f"Predicted Future: {prediction}")

[0.32807183 0.32743254 0.32664907 0.32582557 0.32501212 0.3242294
 0.32348406]
